## Before Starting (If you haven't done it already)
Go to https://aistudio.google.com/app/apikey copy generative language client free tier API key

After you did that click the key icon at the left sidebar add your API key with GOOGLE_API_KEY as name and your API key as the value and enable notebook access

## Setup

### Install dependencies

In [ ]:
%pip install -qU 'google-genai>=1.0.0'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.8/223.8 kB 8.3 MB/s eta 0:00:00


### Set up your API key

To run the following cell, your API key must be stored it in a Colab Secret named `GOOGLE_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see the [Authentication](../quickstarts/Authentication.ipynb) quickstart for an example.

In [ ]:
from google import genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=GOOGLE_API_KEY)

### Choose a model

Different models have different ups and downs.

In [ ]:
MODEL_ID="gemini-2.0-flash" # @param ["gemini-2.5-flash", "gemini-2.5-pro", "gemini-2.0-flash", "gemini-2.5-flash-lite-preview-06-17", ""] {"allow-input":true, isTemplate: true}

## Clear and Direct Instructions for LLMs

Language models respond best to clear and direct instructions.

Think of an LLM like a new hire on their first day. It has no background context on your task beyond what you explicitly provide. Just as you would carefully explain a new task to a person unfamiliar with it, the more clearly and precisely you spell out your expectations, the better the model's performance will be.

---

### The Golden Rule of Clear Prompting

**When in doubt, test your prompt on a human.**  
Share your prompt with a colleague or friend and ask them to follow the instructions exactly. If they’re confused or interpret the task differently than you intended, the model will likely be confused too.

Clarity in your prompt is essential to clarity in the model’s output.

---


### Define Your Goal

Before writing your prompt, clearly define the task and what success looks like. This includes:

- The specific behavior or function you want from the model
- The required format of the output (e.g., single sentence, bullet list, JSON)
- The intended audience or usage context
- Constraints such as tone, length, or domain
- What kinds of outputs would be considered incorrect or off-target


In [ ]:
academic_paragraph = """
A recent study analyzing over 10,000 patient records found that implementing a machine learning triage system in emergency
departments reduced average patient wait times by 28%, compared to traditional manual assessment methods. Furthermore,
the percentage of patients receiving care within the recommended 15-minute window increased from 62% to 85%. The intervention
also reduced instances of critical mis-triaging by 40%, according to post-deployment audits. While the cost of system
integration was approximately $2.5 million, the projected annual savings from efficiency gains and reduced malpractice
claims are estimated at $4.1 million. Researchers conclude that the triage system significantly improves both patient
outcomes and operational efficiency in high-volume clinical environments.
"""


In [ ]:
vague_instruction = "Summarize the text."

chat_vague = client.chats.create(
    model=MODEL_ID,
    config={
        "system_instruction": vague_instruction,
    }
)

response_vague = chat_vague.send_message(academic_paragraph)
print("Vague Output:\n", response_vague.text)


Vague Output:
 A study of over 10,000 patient records found that a machine learning triage system in emergency departments reduced average wait times by 28% and increased the percentage of patients seen within 15 minutes from 62% to 85%. The system also decreased critical mis-triaging by 40%. Despite an initial integration cost of $2.5 million, the system is projected to save $4.1 million annually through increased efficiency and fewer malpractice claims, leading to improved patient outcomes and operational efficiency.


In [ ]:
clear_instruction = (
    "You are a helpful assistant that extracts the **central quantitative claim** from a paragraph "
    "describing an intervention or experiment. Your response must be **one complete sentence**, "
    "and include the most significant measured improvement or impact, using the reported numbers. "
    "Do **not** include costs, side notes, or secondary effects. Focus only on the main numeric result."
)

chat_clear = client.chats.create(
    model=MODEL_ID,
    config={
        "system_instruction": clear_instruction,
    }
)

response_clear = chat_clear.send_message(academic_paragraph)
print("Clear Output:\n", response_clear.text)


Clear Output:
 The machine learning triage system reduced average patient wait times by 28%.


## System Instruction vs. Starting Prompt

**System Instruction** defines the model’s role, tone, and behavior for the entire session.  
It sets the general style and expectations for how the model should respond. This is usually not visible to the end-user.

**Starting Prompt** (or user prompt) is the specific instruction or question given at inference time.  
It tells the model what to do *right now*, often including context, constraints, or examples.

### Example

- **System Instruction**:  
  *"You are a concise and knowledgeable Python programming assistant who explains concepts clearly and avoids unnecessary jargon."*

- **Starting Prompt**:  
  *"Explain the difference between a Python list and a NumPy array, including performance implications and example use cases."*

Use **system instructions** to define the model's persona or function.  
Use **starting prompts** to define the immediate task or question.


## Exercise 1.1 — Write a Story

**Objective:** Modify the prompt in the `story_prompt` variable so that the language model produces a long, rich story.

- Your goal is to make the model generate **at least 800 words**.
- If your answer is sufficiently long, the output cell will turn **green**.
- Tip: The more detailed and specific your prompt, the longer and more coherent the story.




In [ ]:
# Baseline prompt — modify this to increase story length
story_prompt = (
    "#FILL HERE"
)
system_instruction= "#FILL HERE"

In [ ]:
chat_story = client.chats.create(
    model=MODEL_ID,
    config={
        "system_instruction": system_instruction,
    }
)

story_response = chat_story.send_message(story_prompt)
story_text = story_response.text

print("Model Output:\n")
print(story_text)


Model Output:

Sure, I can help you with that! To give you the best possible answer, I need a bit more information.

**What is it you'd like me to fill in?**

Please tell me what you want me to do or what kind of information you're looking for. For example, you could say:

*   "Fill in the blanks in this sentence: 'The cat sat on the ______.'"
*   "Fill in the missing steps in this recipe."
*   "Fill in the details about the history of the Eiffel Tower."
*   "Fill in the code for a simple Python function."
*   "Fill in the next number in this sequence: 2, 4, 6, 8, ____"

The more specific you are, the better I can assist you!


In [ ]:
from IPython.display import display, Markdown

word_count = len(story_text.split())

if word_count >= 800:
    display(Markdown(f"<span style='color:green'><strong>✅ Success:</strong> Story length is {word_count} words.</span>"))
else:
    display(Markdown(f"<span style='color:red'><strong>❌ Too Short:</strong> Story is only {word_count} words. Try adding more details to your prompt.</span>"))


<span style='color:red'><strong>❌ Too Short:</strong> Story is only 121 words. Try adding more details to your prompt.</span>

## Using Examples (Few-shot / Multishot Prompting) to Guide LLM Behavior

Examples are a powerful way to guide language models toward producing structured, high-quality, and task-specific outputs.  
By including 2–5 carefully constructed examples in your prompt, you can significantly improve both the **accuracy** and **consistency** of the model's responses.

This technique—commonly referred to as **few-shot** or **multishot prompting**—is especially effective when you're asking the model to follow a particular format, label categories, or mimic a specific reasoning pattern.

---

### Why Use Examples?

- **Accuracy**: Examples reduce ambiguity and help the model better understand your expectations.
- **Consistency**: Models are more likely to repeat the structure, style, and formatting of your examples.
- **Performance**: Examples improve generalization, especially for edge cases or non-obvious instructions.

---

### Best Practices for Crafting Effective Examples

- **Relevant**: Each example should closely match the real inputs the model will encounter.
- **Diverse**: Cover common cases as well as edge cases or subtle variations to avoid overfitting to a narrow template.
- **Clear**: Use explicit formatting, such as labeled fields or structured delimiters, so the model knows what to follow.

You can even ask the model to review or generate examples for you, if you're unsure whether yours are strong enough.

In [ ]:
fewshot_instruction =""" You are a content moderation assistant. Your task is to read a movie review and label its overall sentiment as either “Positive” or “Negative.” Respond with exactly one word: Positive or Negative.

Example 1:
Review: “An absolute masterpiece—brilliant performances, stunning visuals, and a story that stayed with me long after the credits.”
Sentiment: Positive

Example 2:
Review: “I couldn’t finish it. The plot made no sense, and the acting felt wooden. A total waste of two hours.”
Sentiment: Negative

Example 3:
Review: “Fun in parts, but it dragged in the middle. Some laughs, some groans—mixed bag, really.”
Sentiment: Negative

Example 4:
Review: “Surpassed all my expectations. A heartfelt journey with nuanced characters and a fantastic score.”
Sentiment: Positive

Now classify this new review:
Review: “”
Sentiment:  """

In [ ]:
chat_fewshot = client.chats.create(
    model=MODEL_ID,
    config={
        "system_instruction": fewshot_instruction,
    }
)

sentiment_example="A visually dazzling, yet an emotionally hollow experience."

response_fewshot = chat_fewshot.send_message(sentiment_example)
print("Fewshot Output:\n", response_fewshot.text)


Fewshot Output:
 Negative


## Exercise 1.2 — Multi-Label Email Classification

**Objective**  
Transform the baseline script so that the language model **correctly assigns every relevant category letter** to each email. A prediction is only counted as correct if it matches the ground-truth set **exactly** (order doesn’t matter).

**Dataset**

Ten customer emails of varying tone and content.  
Possible labels (multiple may apply):

| Letter | Meaning                           |
|--------|-----------------------------------|
| **A**  | Pre-sale question                 |
| **B**  | Broken / defective item           |
| **C**  | Billing question                  |
| **D**  | Technical support / usage help    |
| **E**  | Return or refund request          |
| **F**  | Abusive / angry complaint         |
| **G**  | Other       

In [ ]:
import re
from IPython.display import display, Markdown

# --------------------------- Data for the Exercise ---------------------------
# (Data is the same as in your notebook)
EMAILS = [
    "Hi there! I'm considering buying the Mixmaster4000, but I'm vegan – are all the parts dishwasher‑safe?",
    "My brand‑new Mixmaster4000 makes a loud grinding sound and emits smoke. I'd like a full refund, please.",
    "WHY WAS I CHARGED TWICE THIS MONTH?! On top of that, the stupid thing died after a week. SORT IT OUT NOW!!!",
    "Can I use the Mixmaster4000 to make cold‑process soap, or will that void the warranty?",
    "The speed dial sticks between settings 3 and 4. Any way to recalibrate it myself?",
    "I accidentally mailed my phone to you instead of my warranty card. Could you send it back?",
    "I'd like to cancel my order and have the pending charge reversed.",
    "This garbage mixer shredded its own blades and nearly took my hand off – unbelievable trash!",
    "Your site says I have an outstanding balance, but my bank shows the payment cleared last week.",
    "How did I end up on this newsletter list? Also, can my Mixmaster4000 grind almonds into flour?",
]

ANSWERS = [
    ["A"], ["B", "E"], ["B", "C", "F"], ["D"], ["D"],
    ["G"], ["E"], ["B", "F"], ["C"], ["D", "G"],
]

# --- REVISED PROMPT STRUCTURE ---

# 1. The System Instruction now clearly defines the task, categories, and output format.
SYSTEM_INSTRUCTION = """
You are an expert classifier of sentiment classify emails about the product according to the table below:
Emails can have multiple classifications assoicated with them

Letter	Meaning
A	Pre-sale question
B	Broken / defective item
C	Billing question
D	Technical support / usage help
E	Return or refund request
F	Abusive / angry complaint
G	Other
"""

# 2. We provide concrete few-shot examples that will be part of the prompt.
# These examples cover simple, complex, and edge-case scenarios.
FEWSHOT_EXAMPLES = """
What is this horrible product! Your are a bunch of useless people. Give me a full refund for it!
Classification: F, E
"""

# 3. The final prompt template combines instructions, examples, and the new email.
FINAL_PROMPT_TEMPLATE = (
    f"{FEWSHOT_EXAMPLES}\n\n---\n\nNow classify the following email.\n\n"
    'Email: "{email}"\nAnswer:'
)

# --- REVISED AND EFFICIENT CLASSIFICATION LOGIC ---

# Create ONE chat session for the entire task. This is much more efficient.
# Use a low temperature for deterministic, consistent classification.
chat = client.chats.create(
    model=MODEL_ID,
    config={
        "system_instruction": SYSTEM_INSTRUCTION,
        "temperature": 0.1
    }
)

def classify_email(email: str) -> str:
    """Sends a new email to the existing chat session for classification."""
    prompt = FINAL_PROMPT_TEMPLATE.format(email=email)
    return chat.send_message(prompt).text.strip()

# --- RUN THE EVALUATION ---
for i, email in enumerate(EMAILS, 1):
    raw = classify_email(email)
    predicted = set(re.findall(r"[A-G]", raw.upper()))
    correct = set(ANSWERS[i-1])
    ok = predicted == correct

    print("\n" + "-"*90)
    print(f"Email {i}: {email}")
    print("Model raw output :", raw)
    print("Predicted labels :", sorted(list(predicted)) if predicted else "(none)")
    print("Expected labels  :", sorted(correct))
    display(Markdown(
        "<span style='color:green'><strong>✅ Correct</strong></span>"
        if ok else
        "<span style='color:red'><strong>❌ Incorrect</strong></span>"
    ))
    print("-"*90)


------------------------------------------------------------------------------------------
Email 1: Hi there! I'm considering buying the Mixmaster4000, but I'm vegan – are all the parts dishwasher‑safe?
Model raw output : A
Predicted labels : ['A']
Expected labels  : ['A']


<span style='color:green'><strong>✅ Correct</strong></span>

------------------------------------------------------------------------------------------

------------------------------------------------------------------------------------------
Email 2: My brand‑new Mixmaster4000 makes a loud grinding sound and emits smoke. I'd like a full refund, please.
Model raw output : B, E
Predicted labels : ['B', 'E']
Expected labels  : ['B', 'E']


<span style='color:green'><strong>✅ Correct</strong></span>

------------------------------------------------------------------------------------------

------------------------------------------------------------------------------------------
Email 3: WHY WAS I CHARGED TWICE THIS MONTH?! On top of that, the stupid thing died after a week. SORT IT OUT NOW!!!
Model raw output : C, B, F
Predicted labels : ['B', 'C', 'F']
Expected labels  : ['B', 'C', 'F']


...

------------------------------------------------------------------------------------------

------------------------------------------------------------------------------------------
Email 9: Your site says I have an outstanding balance, but my bank shows the payment cleared last week.
Model raw output : C
Predicted labels : ['C']
Expected labels  : ['C']


<span style='color:green'><strong>✅ Correct</strong></span>

------------------------------------------------------------------------------------------

------------------------------------------------------------------------------------------
Email 10: How did I end up on this newsletter list? Also, can my Mixmaster4000 grind almonds into flour?
Model raw output : G, D
Predicted labels : ['D', 'G']
Expected labels  : ['D', 'G']


<span style='color:green'><strong>✅ Correct</strong></span>

------------------------------------------------------------------------------------------


## Chain of thought prompting

Sometimes LLMs can return non-satisfactory answers. To simulate that behavior, you can implement a phrase like "Return the answer immediately" in your prompt.

Without this, the model sometimes uses chain of thought by itself, but it is inconsistent and does not always result in the correct answer.

In [ ]:
prompt = """
  5 people can create 5 donuts every 5 minutes. How much time would it take
  25 people to make 100 donuts? Return the answer immediately.
"""

response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
)
print(response.text)

5 minutes



To influence this you can implement chain of thought into your prompt and look at the difference in the response. Note the multiple steps within the prompt.

In [ ]:
prompt = """
  5 people can create 5 donuts every 5 minutes. How much time would it take
  25 people to make 100 donuts? Return the answer after thinking step by step.
"""

response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
)
print(response.text)

Here's how we can solve this problem step-by-step:

*   **Donut creation rate per person:** If 5 people make 5 donuts in 5 minutes, that means each person makes 1 donut in 5 minutes.

*   **Donut creation rate for 25 people:** If each person makes 1 donut in 5 minutes, then 25 people can make 25 donuts in 5 minutes.

*   **Time to make 100 donuts:** Since 25 people can make 25 donuts in 5 minutes, they can make 100 donuts (25 * 4) in 20 minutes (5 * 4).

**Answer:** It would take 25 people 20 minutes to make 100 donuts.


### Self-Ask prompting

Self-Ask prompting is a relevant method for function calling, especially when a model needs to handle complex queries that require intermediate reasoning steps or external tools. By breaking down a user's question into sub-questions and explicitly deciding which should be answered internally versus via external functions, Self-Ask enables more structured decision-making.

In [ ]:
from IPython.display import Markdown

prompt = """
  Question: Who was the president of the united states when Mozart died?
  Are follow up questions needed?: yes.
  Follow up: When did Mozart died?
  Intermediate answer: 1791.
  Follow up: Who was the president of the united states in 1791?
  Intermediate answer: George Washington.
  Final answer: When Mozart died George Washington was the president of the USA.

  Question: Where did the Emperor of Japan, who ruled the year Maria
  Skłodowska was born, die?
"""

response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt
)

Markdown(response.text)

Are follow up questions needed?: Yes.
Follow up: When was Maria Skłodowska born?
Intermediate answer: 1867.
Follow up: Who was the Emperor of Japan in 1867?
Intermediate answer: Emperor Kōmei.
Follow up: When and where did Emperor Kōmei die?
Intermediate answer: Emperor Kōmei died on January 30, 1867 in Kyoto, Japan.
Final answer: Emperor Kōmei, who ruled Japan the year Maria Skłodowska was born, died in Kyoto, Japan.


## Prompting in structure

First, start by extracting all the groceries. To dod this, set the system instructions when defining the model

In [ ]:
from google.genai import types


MODEL_ID="gemini-2.5-flash"

groceries_system_prompt = f"""
  Your task is to extract to a list all the groceries with its quantities based on the provided recipe.
  Make sure that groceries are in the order of appearance.
"""

grocery_extraction_config =  types.GenerateContentConfig(
    system_instruction=groceries_system_prompt
)

Next, the recipe is defined. You will pass the recipe into `generate_content`, and see that the list of groceries was successfully extracted from the input.

In [ ]:
recipe = """
  Step 1:
  Grind 3 garlic cloves, knob of fresh ginger, roughly chopped, 3 spring onions to a paste in a food processor.
  Add 2 tbsp of clear honey, juice from one orange, 1 tbsp of light soy sauce and 2 tbsp of vegetable oil, then blend again.
  Pour the mixture over the cubed chicken from 4 small breast fillets and leave to marnate for at least 1hr.
  Toss in the 20 button mushrooms for the last half an hour so the take on some of the flavour, too.

  Step 2:
  Thread the chicken, 20 cherry tomatoes, mushrooms and 2 large red peppers onto 20 wooden skewers,
  then cook on a griddle pan for 7-8 mins each side or until the chicken is thoroughly cooked and golden brown.
  Turn the kebabs frequently and baste with the marinade from time to time until evenly cooked.
  Arrange on a platter, and eat with your fingers.
"""

grocery_list = client.models.generate_content(
    model=MODEL_ID,
    contents=recipe,
    config=grocery_extraction_config
)
print(grocery_list.text)


*   garlic cloves, 3
*   fresh ginger, knob
*   spring onions, 3
*   clear honey, 2 tbsp
*   orange, 1
*   light soy sauce, 1 tbsp
*   vegetable oil, 2 tbsp
*   chicken breast fillets, 4 small
*   button mushrooms, 20
*   cherry tomatoes, 20
*   red peppers, 2 large


The next step is to further format the shopping list based on the ingredients extracted.

In [ ]:
shopping_list_system_prompt = """
  You are given a list of groceries. Complete the following:
  - Organize groceries into categories for easier shopping.
  - List each item one under another with a checkbox [].
"""

shopping_list_config = types.GenerateContentConfig(
    system_instruction=shopping_list_system_prompt
)

Now that you have defined the instructions, you can also decide how you want to format your grocery list. Give the prompt a couple examples, or perform few-shot prompting, so it understands how to format your grocery list.

In [ ]:
from IPython.display import Markdown

shopping_list_prompt = f"""
  Make sure every item is listen in proper categories
  LIST: 3 tomatoes, 1 turkey, 4 tomatoes
  OUTPUT:
  ## VEGETABLES
  - [ ] 7 tomatoes
  ## MEAT
  - [ ] 1 turkey

  LIST: {grocery_list.text}
  OUTPUT:
"""

shopping_list = client.models.generate_content(
    model=MODEL_ID,
    contents=shopping_list_prompt,
    config=shopping_list_config
)

Markdown(shopping_list.text)

## VEGETABLES
- [ ] 20 button mushrooms
- [ ] 20 cherry tomatoes
- [ ] 3 garlic cloves
- [ ] knob fresh ginger
- [ ] 2 large red peppers
- [ ] 3 spring onions

## FRUITS
- [ ] 1 orange

## MEAT & POULTRY
- [ ] 4 small chicken breast fillets

## PANTRY
- [ ] 2 tbsp clear honey
- [ ] 1 tbsp light soy sauce
- [ ] 2 tbsp vegetable oil

# Prompting Best Practices: A Concise Guide

1.  **Be Specific and Direct.**
    The model only knows what you tell it. Clearly define the task, desired format, tone, and constraints. *If a human would be confused, the model will be too.*

2.  **Show, Don't Just Tell (Few-Shot Prompting).**
    Provide 2-5 high-quality examples demonstrating the exact output you want. This is the fastest way to improve accuracy and consistency.

3.  **Separate Role from Task.**
    Use a **System Instruction** to set the model's overall persona and role (e.g., "You are an expert financial analyst"). Use the **User Prompt** for the specific, immediate task.

4.  **Guide Complex Reasoning.**
    For multi-step problems, tell the model to "think step by step." This forces a logical workflow and dramatically reduces errors.

5.  **Use Positive/Negative Constraints.**
    Explicitly tell the model what to do and what *not* to do (e.g., "Do not include your own opinions, stick to facts", "Avoid technical jargon.").

6.  **Iterate.**
    Your first prompt is a draft. Test the output, see where it fails, and refine your instructions. Prompting is a cycle of testing and refinement.